In [ ]:
import sklearn
from pathlib import Path
import tarfile
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from zlib import crc32
from sklearn.model_selection import train_test_split
from pandas.plotting import scatter_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error

In [ ]:
# Custom Transformer
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
# Life Satisfaction Example
data_root = "https://github.com/ageron/data/raw/main/"
lifesat = pd.read_csv(data_root + "lifesat/lifesat.csv")

X_lifesat = lifesat[["GDP per capita (USD)"]].values
y_lifesat = lifesat[["Life satisfaction"]].values

plt.scatter(X_lifesat, y_lifesat)
plt.grid(True)
plt.axis([23_500, 62_500, 4, 9])
plt.show()

model_lifesat = LinearRegression()
model_lifesat.fit(X_lifesat, y_lifesat)

X_new = [[33_442.8]]
y_new = model_lifesat.predict(X_new)
print("Predicted life satisfaction for Puerto Rico:", y_new)

In [ ]:
# Load Housing Data
def load_housing_data():
    path = Path("Datasets/housing.tgz")
    if not path.is_file():
        Path("Datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, path)
        with tarfile.open(path) as housing_path:
            housing_path.extractall(path="Datasets")
    return pd.read_csv(Path("Datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

In [ ]:
housing.info()
print(housing["ocean_proximity"].value_counts())
housing.describe()

In [ ]:
housing.hist(bins=50, figsize=(12, 8))
plt.show()

In [ ]:
# Manual shuffle and split
def shuffle_and_split(data, test_ratio, rng):
    shuffle_indices = rng.permutation(len(data))
    test_split = int(len(data) * test_ratio)
    train_set = shuffle_indices[:test_split]
    test_set = shuffle_indices[test_split:]
    return data.iloc[train_set], data.iloc[test_set]

rng = np.random.default_rng()
train_set_manual, test_set_manual = shuffle_and_split(housing, 0.2, rng)
print("Manual Train:", len(train_set_manual), "Test:", len(test_set_manual))

In [ ]:
# Stable test set via CRC32
def is_id_in_the_test_set(identifier, test_ratio):
    return crc32(np.int64(identifier)) < 2**32 * test_ratio

def split_data(data, test_ratio, id_column):
    ids = data[id_column]
    in_test_set = ids.apply(lambda id_: is_id_in_the_test_set(id_, test_ratio))
    return data.loc[~in_test_set], data.loc[in_test_set]

housing_with_id = housing.copy()
housing_with_id["id"] = housing["longitude"] * 1000 + housing["latitude"]
train_set, test_set = split_data(housing_with_id, 0.2, "id")
print(f"Train size: {len(train_set)}, Test size: {len(test_set)}")

In [ ]:
# Stratified Sampling
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)

housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0, 1.5, 3, 4.5, 6, np.inf],
    labels=[1, 2, 3, 4, 5]
)

cat_count = housing["income_cat"].value_counts().sort_index()
cat_count.plot.bar(rot=0, grid=True)
plt.xlabel("Income category")
plt.ylabel("Number of districts")
plt.show()

strat_train_set, strat_test_set = train_test_split(
    housing, test_size=0.2, stratify=housing["income_cat"], random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

In [ ]:
# Explore and Visualize
housing_train = strat_train_set.drop("median_house_value", axis=1).copy()
housing_labels = strat_train_set["median_house_value"].copy()

lon = housing_train["longitude"]
lat = housing_train["latitude"]

scatter = plt.scatter(lon, lat, s=housing_train["population"] / 100,
                      cmap="jet", c=housing_labels, label="population")
plt.grid(True)
plt.legend()
plt.colorbar(scatter, label="median_housing_value")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()

In [ ]:
# Correlations
cor_coeff = strat_train_set.corr(numeric_only=True)
print(cor_coeff["median_house_value"].sort_values(ascending=False))

attributes = ["median_income", "median_house_value", "housing_median_age", "total_rooms"]
scatter_matrix(strat_train_set[attributes], figsize=(12, 8))
plt.show()

plt.scatter(x=housing_train["median_income"], y=housing_labels, alpha=0.1)
plt.grid(True)
plt.xlabel("median_income")
plt.ylabel("median_house_value")
plt.show()

In [ ]:
# Feature Engineering
housing_train["rooms_per_house"] = housing_train["total_rooms"] / housing_train["households"]
housing_train["bedrooms_ratio"] = housing_train["total_bedrooms"] / housing_train["total_rooms"]
housing_train["people_per_house"] = housing_train["population"] / housing_train["households"]

housing_train_with_labels = housing_train.copy()
housing_train_with_labels["median_house_value"] = housing_labels

corr_matrix = housing_train_with_labels.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

In [ ]:
# Preprocessing
housing_num = housing_train.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
imputer.fit(housing_num)
X_imputed = imputer.transform(housing_num)

housing_tr = pd.DataFrame(X_imputed, columns=housing_num.columns, index=housing_num.index)

housing_cat = housing_train[["ocean_proximity"]]

ordinal_encoder = OrdinalEncoder()
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat)

cat_encoder = OneHotEncoder()
housing_cat_1hot = cat_encoder.fit_transform(housing_cat)

std_scaled = StandardScaler()
housing_num_std_scaled = std_scaled.fit_transform(housing_num)

age_simil_35 = rbf_kernel(housing_num[["housing_median_age"]], [[35]], gamma=0.1)

In [ ]:
# TransformedTargetRegressor
some_new_data = housing_train[["median_income"]].iloc[:5]
model_ttr = TransformedTargetRegressor(LinearRegression(), transformer=StandardScaler())
model_ttr.fit(housing_train[["median_income"]], housing_labels)
predictions_ttr = model_ttr.predict(some_new_data)
print(predictions_ttr)

log_transformer = FunctionTransformer(np.log, inverse_func=np.exp)
log_pop = log_transformer.transform(housing_train[["population"]])

rbf_transformer = FunctionTransformer(rbf_kernel, kw_args=dict(Y=[[35.]], gamma=0.1))
age_simil_35 = rbf_transformer.transform(housing_train[["housing_median_age"]])

sf_coords = 37.7749, -122.41
sf_transformer = FunctionTransformer(rbf_kernel, kw_args=dict(Y=[sf_coords], gamma=0.1))
sf_simil = sf_transformer.transform(housing_train[["latitude", "longitude"]])

In [ ]:
# Pipelines
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

sklearn.set_config(display="diagram")

housing_num_prepared = num_pipeline.fit_transform(housing_num)
df_housing_num_prepared = pd.DataFrame(
    housing_num_prepared,
    columns=num_pipeline.get_feature_names_out(),
    index=housing_num.index
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include=object)),
)

housing_prepared = preprocessing.fit_transform(housing_train)

In [ ]:
# Full Preprocessing Pipeline
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]

def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]

def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )

log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population", "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
],
    remainder=default_num_pipeline
)

In [ ]:
# Train and Evaluate
lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing_train, housing_labels)

housing_predictions = lin_reg.predict(housing_train)

print("First 5 predictions:", housing_predictions[:5].round(-2))
print("First 5 actual:     ", housing_labels.iloc[:5].values)

rmse = mean_squared_error(housing_labels, housing_predictions, squared=False)
print(f"Training RMSE: {rmse:,.0f}")